In [1]:
import numpy as np
import matplotlib.pyplot as plt
from netCDF4 import Dataset,num2date,date2num
from datetime import datetime,timedelta
import pandas as pd

Compute the Reflection Index following Schutte et al. 2025 WCD using ERA5

1. Compute 100 hPa meridional eddy heat flux, $v^{*}T^{*}$
2. Compute regional averages over Canada 45–75° N, 230–280° E & Siberia 45–75° N, 140–200° E
3. Apply a running mean (Messori et al. defined 15d, but shorter windows have similar outcomes)
4. Compute anomalies from daily climatology
5. Compute standardised anomalies by dividing by calendar day standard deviation
6. Then RI = Sib - Can

For the purposes of simplicity, delete 29 Feb.

In [7]:
smooth_window=15 # days for smoothing the heat flux
years=np.arange(1979,2027,1)
nwinters = len(years)-1
months_select = [11,12,1,2,3] # index defined for DJFM, use NDJFMA to allow padding
len_season = np.sum([30,31,31,28,31])

In [ ]:
def calc_reflection(years,smooth_window):

    hf_sib_all = np.full(365,np.nan)
    hf_can_all = np.full(np.shape(hf_sib_all),np.nan)
    time_all = np.full(np.shape(hf_sib_all),dtype='object',fill_value=np.nan)
    
    for y, year in enumerate(years):
        print(year)
       
        # first open the meridional wind file
        nc1 = Dataset(f"era5_v_component_of_wind_{year}.nc", "r")
        lats = nc1.variables['latitude'][:]
        lons = nc1.variables['longitude'][:]
        time = num2date(nc1.variables['time'][:],nc1.variables['time'].units,nc1.variables['time'].calendar,only_use_cftime_datetimes=False)       
        level = nc1.variables['isobaricInhPa'][:]
        # 100 hPa level id
        levidx = np.where(level==100)[0]
        # both domains are across 45-75N so just take this
        latcut = np.where((lats>=45)&(lats<=75))[0]
        lats = lats[latcut]
        wgts = np.cos(np.radians(lats))
        # lon bounds for the two domains
        loncut_sib = np.where((lons>=140)&(lons<=200))[0]
        loncut_can = np.where((lons>=230)&(lons<=280))[0]
        # compute departure of v from zonal mean, vstar
        v = nc1.variables['v'][:,levidx,latcut].squeeze()
        vstar = v-v.mean(axis=-1)[:,:,np.newaxis]
        nc1.close()
        
        # # open temperature, this file has same grid
        nc2 = Dataset(f"era5_temperature_{year}.nc", "r")
        t = nc2.variables['t'][:,levidx,latcut].squeeze()
        # departute of t from zonal mean, tstar
        tstar = t-t.mean(axis=-1)[:,:,np.newaxis]
        nc2.close()
        
        ## heat flux, by multiplying vstar and tstar
        hf=vstar*tstar
        # average in the siberian and canada boxed
        hf_sib = np.average(hf[:,:,loncut_sib].mean(axis=-1),axis=-1,weights=wgts)
        hf_can = np.average(hf[:,:,loncut_can].mean(axis=-1),axis=-1,weights=wgts)
    
        # delete feb 29 if necessary
        if year%4==0:
            leapidx = np.where(time==datetime(year,2,29))[0]
            time = np.delete(time,leapidx)
            hf_sib=np.delete(hf_sib,leapidx)
            hf_can=np.delete(hf_can,leapidx)
            
        if y==0:
            hf_sib_all=hf_sib
            hf_can_all=hf_can
            time_all=time
    
        else:
            hf_sib_all = np.concatenate((hf_sib_all,hf_sib))
            hf_can_all = np.concatenate((hf_can_all,hf_can))
            time_all=np.concatenate((time_all,time))
    
    # smooth
    hf_sib_smooth = pd.Series(hf_sib_all).rolling(window=smooth_window, center=True).mean().to_numpy()
    hf_can_smooth = pd.Series(hf_can_all).rolling(window=smooth_window, center=True).mean().to_numpy()

    # subselect to extended winter
    print("Subselecting to extended winter...")
    # go from the start of winter of first year to end of winter in last year
    tidx = np.array([(t.month in months_select)&(t>=datetime(years[0],months_select[0],1))&(t<datetime(years[-1],months_select[0],1)) for t in time_all])
    hf_sib_smooth = hf_sib_smooth[tidx]
    hf_can_smooth = hf_can_smooth[tidx]
    time_all=time_all[tidx]
    print(time_all[0])
    print(time_all[-1])

    print("Anomaly calc...")    
    # reshape to year, day for anom calc
    hf_sib_smooth =np.reshape(hf_sib_smooth,(nwinters,len_season))
    hf_can_smooth =np.reshape(hf_can_smooth,np.shape(hf_sib_smooth))
    time_all=np.reshape(time_all,np.shape(hf_sib_smooth))
    # compute climatology
    hf_sib_clim=hf_sib_smooth.mean(axis=0)
    hf_can_clim=hf_can_smooth.mean(axis=0)
    
    # compute std dev
    hf_sib_std=hf_sib_smooth.std(axis=0)
    hf_can_std=hf_can_smooth.std(axis=0)
    
    # compute std anom
    hf_sib_std_anom = (hf_sib_smooth-hf_sib_clim)/hf_sib_std
    hf_can_std_anom = (hf_can_smooth-hf_can_clim)/hf_can_std

    print("Computing reflection index...")
    # reflection index is the difference of the two
    ri = hf_sib_std_anom-hf_can_std_anom
    print("Done!")
    
    return ri, time_all,hf_sib_clim,hf_can_clim,hf_sib_std,hf_can_std

In [26]:
ri, time_all,hf_sib_clim,hf_can_clim,hf_sib_std,hf_can_std=calc_reflection(years,smooth_window)

1979
1980
1981
1982
1983
1984
1985
1986
1987
1988
1989
1990
1991
1992
1993
1994
1995
1996
1997
1998
1999
2000
2001
2002
2003
2004
2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026
Subselecting to extended winter...
1979-11-01 00:00:00
2026-03-31 00:00:00
Anomaly calc...
Computing reflection index...
Done!
